# Step 3 (Baseline YOLO) — Colab GPU TrainingNotebook chạy end-to-end (Runtime ▸ **Run all**) để huấn luyện B01 YOLO11n và B02 YOLO11s 100 epoch trên GPU Colab, trích dự đoán calibration/fusion, và tải artifact về máy local.**Trước khi chạy:**1. Upload toàn bộ repo `PCB_Lab_impl3` lên Google Drive (hoặc Git clone vào `/content`).2. Upload `DatasetVer4_Public` vào Drive tại đường dẫn `DATASET_ROOT` (mặc định  `/content/drive/MyDrive/DatasetVer4_Public`). Sửa Cell 2 nếu khác.3. Runtime ▸ Change runtime type ▸ **GPU** (T4/V100/A100).> Quy tắc: dataset chỉ đọc; Ultralytics chỉ thấy `data.yaml` do `build_yolo_view` tạo. Không bao giờ ghi vào `DATASET_ROOT`. Tập `test` bị chặn tuyệt đối.

In [ ]:
# ===== Cell 1: Gate 0 — Hardware Verification =====
import os, sys, subprocess

def banner(t):
    print('=' * 60)
    print(t)
    print('=' * 60)

banner('Gate 0: CUDA hardware check')
!nvidia-smi || echo 'nvidia-smi not found'

import torch
cuda_ok = torch.cuda.is_available()
print('torch.version:', torch.__version__)
print('torch.cuda.is_available():', cuda_ok)
if cuda_ok:
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print('GPU[0]:', name, '(compute capability', cap, ')')
    cc = torch.cuda.get_device_capability(0)
    # Colab CUDA 12.x reports capability >= 7.0 for T4/V100/A100
    assert cc[0] >= 7, f'GPU compute capability {cc} too low for AMP/torch CUDA'
    print('CUDA compute capability OK for training')
else:
    raise SystemExit('Gate 0 FAILED: no CUDA GPU. Change runtime to GPU.')

In [ ]:
# ===== Cell 2: Dataset Setup =====
from pathlib import Path

# Mount Google Drive (click authorize when prompted)
from google.colab import drive
drive.mount('/content/drive')

# Thu muc DatasetVer4_Public tren Drive — sua neu khac
DATASET_ROOT = '/content/drive/MyDrive/DatasetVer4_Public'
os.environ['DATASET_ROOT'] = DATASET_ROOT

manifest = Path(DATASET_ROOT) / 'benchmarks' / 'deeppcb' / 'manifests' / 'samples.jsonl'
assert manifest.exists(), f'Manifest canonical missing: {manifest}\nUpload DatasetVer4_Public to {DATASET_ROOT}'
print('DATASET_ROOT:', DATASET_ROOT)
print('Canonical manifest OK:', manifest)
print('Manifest size (bytes):', manifest.stat().st_size)

In [ ]:
# ===== Cell 3: Install Environment =====
# Colab already provides a CUDA build of torch. Do NOT pip-install the pinned
# torch==2.14.0 CPU wheels (would downgrade and break CUDA). We install the
# remaining pinned deps, skipping torch/torchvision, then editable-install pcb_lab.
import re, sys

req_path = 'requirements/step3.txt'
skip = {'torch', 'torchvision'}
kept = []
for line in open(req_path, encoding='utf-8'):
    s = line.strip()
    if not s or s.startswith('#'):
        continue
    pkg = re.split(r'[=<>!~ ]', s)[0].strip().lower()
    if pkg in skip:
        print('SKIP (use Colab CUDA torch):', s)
        continue
    kept.append(s)

filtered = '/content/step3_filtered.txt'
open(filtered, 'w', encoding='utf-8').write('\n'.join(kept) + '\n')
print('Installing (excluding torch/torchvision):')
for k in kept:
    print('  ', k)

!pip install -r {filtered}
!pip install --no-build-isolation --no-deps -e .

import pcb_lab
print('pcb_lab imported, version:', getattr(pcb_lab, '__version__', 'n/a'))

In [ ]:
# ===== Cell 4: Download Pretrained Weights =====
from pathlib import Path
import urllib.request

PRETRAINED_DIR = Path('artifacts/pretrained')
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)

ASSET_BASE = 'https://github.com/ultralytics/assets/releases/download/v8.3.0'
files = ['yolo11n.pt', 'yolo11s.pt']

for fn in files:
    dst = PRETRAINED_DIR / fn
    if dst.exists() and dst.stat().st_size > 1_000_000:
        print('already present:', dst)
        continue
    url = f'{ASSET_BASE}/{fn}'
    print('downloading', url)
    urllib.request.urlretrieve(url, dst)
    print('  ->', dst, dst.stat().st_size, 'bytes')

print('Pretrained ready:', sorted(p.name for p in PRETRAINED_DIR.glob('*.pt')))

In [ ]:
# ===== Cell 5: Train B01 YOLO11n (100 epochs, seed 42, AdamW, AMP auto-on GPU) =====
# --allow-download: fallback fetch if Cell 4 weights are missing.
# AMP: omitted -> auto-enabled on CUDA GPU (off on CPU). --device 0 = first GPU.
!python scripts/train_yolo.py --config configs/models/yolo11n.yaml --seed 42 --device 0 --allow-download

from pathlib import Path
print('B01 artifact:')
for p in sorted(Path('artifacts/yolo/B01_yolo11n/seed42').iterdir()):
    print('  ', p.name, p.stat().st_size)

In [ ]:
# ===== Cell 6: Train B02 YOLO11s (100 epochs, seed 42, AdamW, AMP auto-on GPU) =====
!python scripts/train_yolo.py --config configs/models/yolo11s.yaml --seed 42 --device 0 --allow-download

from pathlib import Path
print('B02 artifact:')
for p in sorted(Path('artifacts/yolo/B02_yolo11s/seed42').iterdir()):
    print('  ', p.name, p.stat().st_size)

In [ ]:
# ===== Cell 7: Extract Predictions (calibration + fusion only; test is blocked) =====
print('--- B01 YOLO11n ---')
!python scripts/extract_yolo_predictions.py --artifact artifacts/yolo/B01_yolo11n/seed42 --partitions calibration fusion
print('--- B02 YOLO11s ---')
!python scripts/extract_yolo_predictions.py --artifact artifacts/yolo/B02_yolo11s/seed42 --partitions calibration fusion

from pathlib import Path
print('\nExtracted prediction files:')
for p in sorted(Path('artifacts/yolo').rglob('preds_*.jsonl')):
    print('  ', p, p.stat().st_size, 'bytes')

In [ ]:
# ===== Cell 8: Export Bundle (download artifacts back to local) =====
import shutil, os
from pathlib import Path
from google.colab import files

bundle = 'yolo_step3_artifacts.zip'
if Path(bundle).exists():
    os.remove(bundle)
shutil.make_archive('yolo_step3_artifacts', 'zip', 'artifacts/yolo')
print('Created', bundle, Path(bundle).stat().st_size, 'bytes')
print('Contents (top-level model dirs):')
for p in sorted(Path('artifacts/yolo').iterdir()):
    print('  ', p.name)
files.download(bundle)
print('Download triggered. Save yolo_step3_artifacts.zip to your local machine.')